# EcoRewind 2.0 — Phase 0 gate: audit the re-exports

Run after all export tasks are COMPLETED. Reads the GeoTIFFs directly from Drive; writes `data_audit_v2.md` to `MyDrive/EcoRewind2/reports/`. CPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- fill these in -----------------------------------------------------------
PROJECT = 'your-ee-project-id'          # Earth Engine cloud project (code.earthengine.google.com -> project)
import glob, os
ZIP = max(glob.glob('/content/drive/MyDrive/EcoRewind2/ecorewind*.zip'), key=os.path.getmtime)  # newest code zip
print('code:', ZIP)
REPORTS = '/content/drive/MyDrive/EcoRewind2/reports'   # preflight, manifests, audit persist here
# ----------------------------------------------------------------------------
import os
os.environ['ECOREWIND_REPORTS'] = REPORTS
os.makedirs(REPORTS, exist_ok=True)
!rm -rf /content/ecorewind && unzip -q "$ZIP" -d /content
!pip -q install earthengine-api pyproj rasterio==1.4.3
%cd /content/ecorewind
!python -m pytest -q tests

In [ ]:
EXPORTS = '/content/drive/MyDrive/EcoRewind2_exports'
import glob; print(len(glob.glob(f'{EXPORTS}/*.tif')), 'tifs (expect 320)')

Smoke test on 4 files, then the full audit (`STRIDE = 2` reads every 2nd pixel; use 1 for the final report). Cached per file: re-run after a disconnect.

In [ ]:
!python scripts/audit_exports.py --dir "$EXPORTS" --max-files 4 --stride 4 --out /content/smoke

In [ ]:
STRIDE = 2
!python scripts/audit_exports.py --dir "$EXPORTS" --stride $STRIDE
from IPython.display import Markdown
Markdown(open(f'{REPORTS}/data_audit_v2.md').read())